# 06 — Training GPT-2 small, and running OpenAI's real weights in your NumPy code

We have GPT-2 small with verified gradients. In this notebook:

1. GPT-2 small's **training recipe**: every hyperparameter and where it comes from;
2. the **compute and memory budget**, measured on your machine: why a full pretraining run is a GPU job;
3. data: Tiny Shakespeare → GPT-2 tokens (full 50,257 vocabulary) → batches;
4. **AdamW** from scratch, gradient clipping, and the warmup + cosine schedule with GPT-2 small's values;
5. sanity checks at full size: initial loss ≈ 10.98, and overfitting one batch;
6. **real training steps of the full 124M model**, with gradient accumulation;
7. generating text: temperature and top-k;
8. **loading OpenAI's pretrained GPT-2 small weights** into your NumPy model, generating with it, and fine-tuning it on Shakespeare.

**RAM:** about 4 GB (float32 parameters + gradients + two Adam buffers ≈ 2 GB, plus activations).

In [ ]:
import gc, math, os, time, urllib.request
import numpy as np
import tiktoken
from gpt_numpy import GPT, CrossEntropyLoss, load_hf_gpt2

np.set_printoptions(precision=4, suppress=True, linewidth=120)
rng = np.random.default_rng(1337)
enc = tiktoken.get_encoding("gpt2")
loss_fn = CrossEntropyLoss()
DTYPE = np.float32          # full-size training uses float32: half the memory of float64, ~2x faster

## 1. GPT-2 small's training recipe

The GPT-2 paper (Radford et al., 2019) gives the architecture, the 50,257-token vocabulary, the 1,024-token context and a batch of 512 sequences, but not the full optimizer settings. The standard recipe for a model of this size comes from the GPT-3 paper (Brown et al., 2020), whose "GPT-3 Small" has exactly GPT-2 small's shape (12 layers, 12 heads, C = 768). Open-source reproductions of GPT-2 small, such as nanoGPT and llm.c, use these values:

| hyperparameter | value | meaning |
|---|---|---|
| tokens per optimizer step | 524,288 = 2¹⁹ ≈ 0.5M | 512 sequences × 1,024 tokens |
| max learning rate | 6 × 10⁻⁴ | peak of the schedule |
| min learning rate | 6 × 10⁻⁵ | 10% of max, end of cosine decay |
| warmup | 375M tokens = 715 steps | linear ramp from 0 to max |
| AdamW β₁, β₂, ε | 0.9, 0.95, 10⁻⁸ | |
| weight decay | 0.1 | on 2-D tensors only |
| gradient clipping | global norm 1.0 | |
| init | N(0, 0.02²); residual projections 0.02/√24 | notebook 01 |

## 2. The compute and memory budget

**FLOPs per token.** A matmul with an `(in, out)` weight costs 2·in·out FLOPs per token in the forward pass, and about twice that in the backward pass (one product for `dx`, one for `dW`). So training costs roughly **3 × forward**. Attention adds a term that grows with T (computing $QK^\top$ and $AV$).

In [ ]:
V, C, L, T_ctx = 50257, 768, 12, 1024
matmul_params = L * 12 * C * C + V * C                 # weights in every Linear + the tied head
fwd_per_token = 2 * matmul_params + 4 * L * T_ctx * C  # + attention scores and A@V at T = 1024
train_per_token = 3 * fwd_per_token
print(f"weights used in matmuls     {matmul_params:>15,}")
print(f"forward FLOPs per token     {fwd_per_token:>15,}")
print(f"training FLOPs per token    {train_per_token:>15,}  (~{train_per_token / 1e9:.2f} GFLOP)")
print(f"training FLOPs per GPT-2 optimizer step (524,288 tokens): {train_per_token * 524288:.2e}")

Now measure what **your** machine does with our NumPy GPT-2 small: one forward + backward pass at B = 1, T = 128.

In [ ]:
model = GPT(dtype=DTYPE, seed=0)                       # GPT-2 small, randomly initialized
params = model.parameters()
x = rng.integers(0, V, size=(1, 128)); y = rng.integers(0, V, size=(1, 128))
model.zero_grad(); loss_fn.forward(model.forward(x), y); model.backward(loss_fn.backward())   # warm-up
t0 = time.time()
model.zero_grad(); loss_fn.forward(model.forward(x), y); model.backward(loss_fn.backward())
dt = time.time() - t0
tok_per_s = 128 / dt
print(f"forward+backward of 128 tokens: {dt:.2f} s  ->  {tok_per_s:,.0f} tokens/s")
print(f"effective throughput: {tok_per_s * train_per_token / 1e9:.0f} GFLOP/s")
print(f"one real GPT-2 optimizer step (524,288 tokens): {524288 / tok_per_s / 3600:.1f} hours")
print(f"10 billion tokens (a typical GPT-2 small reproduction run): {1e10 / tok_per_s / 3600 / 24 / 365:.1f} years")

That last number is the honest answer to "can I pretrain GPT-2 small in NumPy on a CPU?": the code is correct and complete, but pretraining needs on the order of $10^{19}$ FLOPs. A modern GPU does ~$10^{14}$ FLOP/s, and real runs use many at once. What we *can* do on a CPU:

- run real training steps of the full 124M model, and watch it learn (sections 5–6);
- load the weights OpenAI obtained from the real pretraining run into **our** NumPy code, use them, and fine-tune them (section 8).

**Memory for training in float32:**

In [ ]:
n_params = sum(p.size for _, p, _ in params)
mb = lambda n: n * 4 / 1e6
print(f"parameters                 {mb(n_params):8,.0f} MB")
print(f"gradients                  {mb(n_params):8,.0f} MB")
print(f"AdamW m and v              {mb(2 * n_params):8,.0f} MB")
for B, T in [(1, 128), (2, 256), (8, 1024)]:
    logits3 = 3 * B * T * V                                   # logits, probs, dlogits
    att = L * 12 * B * T * T                                  # cached attention weights, all layers
    other = L * B * T * C * 20                                # rough: other cached activations per layer
    print(f"activations at B={B}, T={T:<5}  ~{mb(logits3 + att + other):8,.0f} MB")

We use small B and T and reach large *effective* batches with gradient accumulation (section 6).

## 3. Data: Tiny Shakespeare in GPT-2 tokens

~1.1M characters of Shakespeare (public domain), tokenized with tiktoken. We use the **full 50,257-token vocabulary**, so the model is exactly GPT-2 small.

In [ ]:
path = "input.txt"
if not os.path.exists(path):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt", path)
text = open(path, encoding="utf-8").read()
data = np.array(enc.encode_ordinary(text), dtype=np.int64)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print(f"{len(text):,} characters -> {len(data):,} tokens ({len(text) / len(data):.2f} chars/token)")
print(f"train {len(train_data):,} tokens | val {len(val_data):,} tokens")
print("first tokens:", [enc.decode([t]) for t in data[:12]])

def get_batch(split, B, T):
    d = train_data if split == "train" else val_data
    ix = rng.integers(0, len(d) - T - 1, size=B)
    x = np.stack([d[i : i + T] for i in ix])            # (B, T) token ids
    y = np.stack([d[i + 1 : i + T + 1] for i in ix])    # (B, T) next tokens
    return x, y

def estimate_loss(model, B=2, T=128, eval_iters=5):
    out = {}
    for split in ["train", "val"]:
        out[split] = float(np.mean([loss_fn.forward(model.forward(x), y)
                                    for x, y in (get_batch(split, B, T) for _ in range(eval_iters))]))
    return out

## 4. The optimizer, clipping and schedule

### AdamW

**Why not plain SGD (`p -= lr * g`)?** Different parameters have gradients of wildly different magnitudes (embedding rows of rare tokens, LayerNorm gains, attention weights), so one global learning rate is too big for some and too small for others. Mini-batch gradients are also noisy.

**Adam** keeps two exponential moving averages per parameter, elementwise:

$$
m_t = \beta_1 m_{t-1} + (1-\beta_1)\,g_t \quad\text{(momentum: smooths noise)}, \qquad
v_t = \beta_2 v_{t-1} + (1-\beta_2)\,g_t^2 \quad\text{(typical squared magnitude)}
$$

**Bias correction.** $m_0 = v_0 = 0$, and unrolling gives $\mathbb{E}[m_t] = (1-\beta_1^t)\,\mathbb{E}[g]$, so early averages are too small. Dividing fixes it: $\hat m_t = m_t/(1-\beta_1^t)$, $\hat v_t = v_t/(1-\beta_2^t)$.

**Update:** $\theta \leftarrow \theta - \eta\,\hat m_t / (\sqrt{\hat v_t}+\epsilon)$. Since $\hat m/\sqrt{\hat v}$ is roughly "the gradient divided by its own typical size", **every parameter moves by about $\eta$ per step**, whatever its raw gradient scale, and less when its gradient keeps flipping sign.

**Decoupled weight decay (the "W").** AdamW applies $\theta \leftarrow \theta - \eta\lambda\theta$ directly, rather than adding $\lambda\theta$ to the gradient (where it would get divided by $\sqrt{\hat v}$ and distorted). It is applied only to **2-D tensors**: all matmul weights and both embeddings. Biases and LayerNorm gains are left alone.

**Memory:** $m$ and $v$ each have the shape of the parameters, so AdamW adds 2 × 124M numbers. **In place:** every update is `p -= ...` on the same arrays the layers compute with.

In [ ]:
class AdamW:
    def __init__(self, params, betas=(0.9, 0.95), eps=1e-8, weight_decay=0.1):
        self.params = params
        self.b1, self.b2 = betas
        self.eps, self.wd = eps, weight_decay
        self.m = [np.zeros_like(p) for _, p, _ in params]
        self.v = [np.zeros_like(p) for _, p, _ in params]
        self.t = 0

    def step(self, lr):
        self.t += 1
        c1, c2 = 1 - self.b1 ** self.t, 1 - self.b2 ** self.t      # bias corrections (Python floats)
        for i, (_, p, g) in enumerate(self.params):
            self.m[i] *= self.b1; self.m[i] += (1 - self.b1) * g    # in-place EMA updates
            self.v[i] *= self.b2; self.v[i] += (1 - self.b2) * g * g
            if p.ndim >= 2:
                p -= lr * self.wd * p                                # decoupled weight decay
            p -= lr * (self.m[i] / c1) / (np.sqrt(self.v[i] / c2) + self.eps)

print("decayed (2-D) parameters:    ", f"{sum(p.size for _, p, _ in params if p.ndim >= 2):,}")
print("not decayed (biases, LN):    ", f"{sum(p.size for _, p, _ in params if p.ndim < 2):,}")

### Gradient clipping by global norm

Occasionally one batch produces a huge gradient, and one giant step can wreck a model. Compute the norm over **all 124M gradient entries**, $\lVert g\rVert = \sqrt{\sum g^2}$, and if it exceeds 1.0, scale every gradient by $1.0/\lVert g\rVert$. This keeps the direction and caps the length. Logging this norm is one of the best health signals during training.

In [ ]:
def clip_grad_norm(params, max_norm):
    total = math.sqrt(sum(float(np.dot(g.ravel(), g.ravel())) for _, _, g in params))
    if total > max_norm:
        for _, _, g in params:
            g *= max_norm / (total + 1e-6)
    return total

### Learning-rate schedule: GPT-2 small's values

Linear **warmup** over 715 steps to 6 × 10⁻⁴: early on, Adam's $\hat v$ is estimated from few gradients and the weights are random, so small steps first. Then **cosine decay** to 6 × 10⁻⁵: big steps make fast progress early, and small steps late let the model settle. With 10B tokens at 524,288 per step, a run is 19,073 steps.

In [ ]:
def get_lr(it, max_lr=6e-4, min_lr=6e-5, warmup=715, max_steps=19073):
    if it < warmup:
        return max_lr * (it + 1) / warmup
    if it >= max_steps:
        return min_lr
    progress = (it - warmup) / (max_steps - warmup)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))

for it in [0, 357, 714, 715, 5000, 10000, 15000, 19072]:
    print(f"step {it:6d}  lr {get_lr(it):.2e}")

## 5. Sanity checks at full size

### 5.1 Initial loss

Expected for GPT-2 small's initialization: $\ln 50257 + 768 \cdot 0.02^2/2 \approx 10.98$ (derived in notebook 02). A single batch lands within a few tenths of it; 15 or 50 would mean a bug.

In [ ]:
x, y = get_batch("train", 2, 128)
print(f"initial loss {loss_fn.forward(model.forward(x), y):.3f}   expected ~ {np.log(V) + C * 0.02**2 / 2:.3f}")

### 5.2 Overfit one batch

A correct model plus optimizer must be able to memorize a single fixed batch. If the loss doesn't collapse, something is broken (gradients, in-place updates, the optimizer). We use the real 124M model, one sequence of 64 tokens, and a constant learning rate. Expect the loss to fall from ~11 to around 1–2 within 20 steps; an occasional upward jump in the first few steps is normal for Adam at this learning rate.

In [ ]:
optimizer = AdamW(params, weight_decay=0.0)
xb, yb = get_batch("train", 1, 64)
t0 = time.time()
for step in range(20):
    model.zero_grad()
    loss = loss_fn.forward(model.forward(xb), yb)
    model.backward(loss_fn.backward())
    optimizer.step(lr=3e-4)
    print(f"step {step:3d}  loss {loss:7.4f}   ({time.time() - t0:5.1f}s)")

## 6. Training the full GPT-2 small on Shakespeare

A fresh model, the real optimizer settings, gradient accumulation, clipping, and a schedule. Only the **number of steps and the batch size** are scaled down to fit a CPU; the architecture, initialization, β's, weight decay and clipping are GPT-2 small's. Warmup is shortened in proportion to the run.

### Gradient accumulation

GPT-2's 524,288-token batch doesn't fit in memory at once, even on GPUs. Split it into `k` micro-batches, run forward/backward on each **without zeroing gradients in between** (the `+=` accumulates them), then take one optimizer step. Each micro-batch's `dlogits` is divided by `k`, so the accumulated gradient is exactly the gradient of the average loss over all `k·B·T` tokens. Here: B = 2, T = 128, k = 2, so 512 tokens per step (GPT-2 uses k = 524,288 / (B·T)).

In [ ]:
del optimizer, model, params; gc.collect()
model = GPT(dtype=DTYPE, seed=0)
params = model.parameters()
optimizer = AdamW(params, betas=(0.9, 0.95), eps=1e-8, weight_decay=0.1)

B, T, grad_accum_steps = 2, 128, 2
max_steps, warmup = 40, 4
history = []
t0 = time.time()
for step in range(max_steps):
    lr = get_lr(step, warmup=warmup, max_steps=max_steps)
    model.zero_grad()                                              # 1. clear gradients
    loss_accum = 0.0
    for micro in range(grad_accum_steps):
        x, y = get_batch("train", B, T)                            # 2. data
        loss = loss_fn.forward(model.forward(x), y)                # 3-4. forward + loss
        model.backward(loss_fn.backward() / grad_accum_steps)      # 5. backward, grads accumulate
        loss_accum += loss / grad_accum_steps
    norm = clip_grad_norm(params, 1.0)                             # 6. clip
    optimizer.step(lr)                                             # 7. update in place
    history.append(loss_accum)
    if step % 5 == 0 or step == max_steps - 1:
        print(f"step {step:3d} | loss {loss_accum:.4f} | grad norm {norm:6.3f} | lr {lr:.2e} | {time.time() - t0:6.1f}s")

losses = estimate_loss(model)
print(f"\nafter {max_steps} steps: train {losses['train']:.3f} | val {losses['val']:.3f}")

The loss falls well below the initial ≈ 10.98 within a few dozen steps. The model first learns which tokens are frequent (`'\n'`, `','`, `' the'`, `':'`), then common pairs. Getting to fluent text takes billions of tokens, which is section 2's budget problem. Training longer here (more steps, larger B and T) keeps improving it.

In [ ]:
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(7, 3))
    plt.plot(history, marker=".")
    plt.axhline(np.log(V) + C * 0.02**2 / 2, ls="--", c="gray", label="initial loss ~10.98")
    plt.xlabel("optimizer step"); plt.ylabel("train loss"); plt.legend(); plt.show()
except ImportError:
    print("matplotlib not installed - skipping the plot")

## 7. Generating text

Generation is autoregressive: feed the context, take the logits at the **last position only**, turn them into probabilities, sample one token, append it, repeat. If the context grows past 1,024 tokens, keep only the last 1,024.

- **Temperature** τ divides the logits: τ < 1 is sharper and safer, τ > 1 is more random.
- **Top-k** keeps the k most likely tokens and renormalizes. GPT-2 samples are commonly drawn with k = 40 or 50.

(This recomputes the whole context for every new token. Real implementations cache each layer's keys and values, a "KV cache", so each new token costs O(T) instead of O(T²). The outputs are the same.)

In [ ]:
def generate(model, prompt, max_new_tokens, temperature=1.0, top_k=50, num_samples=1):
    idx = np.array([enc.encode(prompt)] * num_samples)
    for _ in range(max_new_tokens):
        logits = model.forward(idx[:, -model.block_size:])[:, -1, :].astype(np.float64) / temperature
        if top_k is not None:
            kth = np.sort(logits, axis=-1)[:, [-top_k]]
            logits = np.where(logits < kth, -np.inf, logits)
        probs = np.exp(logits - logits.max(axis=-1, keepdims=True))
        probs /= probs.sum(axis=-1, keepdims=True)
        nxt = np.array([[rng.choice(len(p), p=p)] for p in probs])
        idx = np.concatenate([idx, nxt], axis=1)
    return [enc.decode(list(row)) for row in idx]

print(generate(model, "ROMEO:", 30, temperature=0.8, top_k=40)[0])

After only a few dozen steps, expect frequent tokens, line breaks and punctuation rather than sentences.

## 8. OpenAI's pretrained GPT-2 small, running in your NumPy code

Our parameter layout matches OpenAI's GPT-2 exactly. HuggingFace stores the weights of GPT-2's `Conv1D` layers as `(in, out)`, the same as our `x @ W`, so loading is a direct copy with no transposes:

| HuggingFace | ours |
|---|---|
| `wte.weight`, `wpe.weight` | `wte.W`, `wpe.W` |
| `h.{i}.ln_1.weight` / `.bias` | `h.{i}.ln_1.g` / `.b` |
| `h.{i}.attn.c_attn.weight` / `.bias` | `h.{i}.attn.c_attn.W` / `.b` |
| `h.{i}.attn.c_proj.*`, `h.{i}.ln_2.*`, `h.{i}.mlp.c_fc.*`, `h.{i}.mlp.c_proj.*` | same pattern |
| `ln_f.weight` / `.bias` | `ln_f.g` / `.b` |
| `h.{i}.attn.bias` | skipped: a stored causal mask, not a parameter |

`load_hf_gpt2` in `gpt_numpy.py` reads the `.safetensors` file with plain NumPy and asserts that every shape matches. The download is 548 MB, once.

In [ ]:
del optimizer; gc.collect()                       # free the Adam buffers (1 GB) before loading
weights = "gpt2_small.safetensors"
url = "https://huggingface.co/openai-community/gpt2/resolve/main/model.safetensors"
pretrained = False
try:
    if not os.path.exists(weights):
        print("downloading GPT-2 small weights (548 MB) ...")
        urllib.request.urlretrieve(url, weights)
    n_loaded = load_hf_gpt2(model, weights)           # copies into the SAME arrays, in place
    pretrained = True
    print(f"loaded {n_loaded} tensors into our NumPy GPT-2 small")
except Exception as e:
    print("could not download/load the weights:", e)
    print("download", url, "manually, save it as", weights, "next to this notebook, and re-run this cell")

In [ ]:
if pretrained:
    for s in generate(model, "Hello, I'm a language model,", 30, temperature=1.0, top_k=50, num_samples=3):
        print(s, "\n---")

This is GPT-2 small, the model OpenAI released in 2019, running entirely on the NumPy code you derived. Every number passes through your `Linear`, `GELU`, `LayerNorm`, `CausalSelfAttention` and tied head.

### How good is it on Shakespeare, without ever training on it?

In [ ]:
if pretrained:
    losses = estimate_loss(model, B=2, T=256, eval_iters=5)
    print(f"pretrained GPT-2 small:   train {losses['train']:.3f} | val {losses['val']:.3f}")
    print("(random initialization: ~10.98)")

### Fine-tuning it on Shakespeare

Fine-tuning is just training that starts from pretrained weights instead of random ones, usually with a smaller learning rate, so the model adapts without forgetting what it knows. The same loop as section 6:

In [ ]:
if pretrained:
    params = model.parameters()
    optimizer = AdamW(params, weight_decay=0.1)
    t0 = time.time()
    for step in range(20):
        model.zero_grad()
        x, y = get_batch("train", 2, 128)
        loss = loss_fn.forward(model.forward(x), y)
        model.backward(loss_fn.backward())
        norm = clip_grad_norm(params, 1.0)
        optimizer.step(lr=3e-5)
        if step % 5 == 0 or step == 19:
            print(f"step {step:3d} | loss {loss:.4f} | grad norm {norm:6.3f} | {time.time() - t0:6.1f}s")
    losses = estimate_loss(model, B=2, T=256, eval_iters=5)
    print(f"\nafter fine-tuning:  train {losses['train']:.3f} | val {losses['val']:.3f}")
    print(generate(model, "ROMEO:", 40, temperature=0.8, top_k=40)[0])

## 9. Saving and loading your weights

Parameter names make a natural key format. Loading must write **into the existing arrays** (`p[...] = ...`), for the in-place reason from notebook 00. A GPT-2 small checkpoint is ~500 MB in float32, so this is off by default.

In [ ]:
SAVE_CHECKPOINT = False
if SAVE_CHECKPOINT:
    np.savez("gpt2_small_numpy.npz", **{name: p for name, p, _ in model.parameters()})
    restored = GPT(dtype=DTYPE, seed=123)
    ckpt = np.load("gpt2_small_numpy.npz")
    for name, p, _ in restored.parameters():
        p[...] = ckpt[name]
    x, y = get_batch("val", 1, 64)
    print("same loss after reload:", np.isclose(loss_fn.forward(model.forward(x), y),
                                                loss_fn.forward(restored.forward(x), y)))

## Where to go from here

- **KV cache** for fast generation: store each layer's `k` and `v` and compute only the new position.
- **GPT-2 medium / large / XL**: `GPT(n_layer=24, n_head=16, n_embd=1024)` and so on, with the matching HuggingFace files (`openai-community/gpt2-medium`, ...). The code doesn't change.
- **Rewrite your own script** with these pieces. The bugs found along the way are listed at the end of notebooks 01, 04 and 05.

## The whole series in one picture

```
text --tiktoken--> (B,T) ids --wte+wpe--> (B,T,768) --12 x [LN->attn(12x64)->+ , LN->MLP(3072)->+]--> ln_f
     --> @ wte.T --> (B,T,50257) logits --CE--> loss (~10.98 at init)
                                                  |
        (probs - onehot)/N  <---------------------+
              |
   tied head -> ln_f -> 12 blocks reversed -> wte scatter-add + wpe sum over B
              |
   124,439,808 gradients --clip at 1.0--> AdamW(0.9, 0.95, wd 0.1) --> repeat
```